### Find r_2 again for purt_course2.yaml

In [38]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from race_env import RaceEnv, point_segment_distance

COURSE_PATH = "../../ros2_ws/src/pylon_2026_world/purt_course2.yaml"
course = RaceEnv(COURSE_PATH)
G = course.GATES
n = course.n_gates
PYLON_FIELD_R_M = course.PYLON_FIELD_R_M

START_MAP = np.array([-15.0, -7.5])


def gate_distance(xy, k):
    return float(point_segment_distance(xy, G[k]["p1"], G[k]["p2"]))


def gate_points(k, n_pts=2001, clearance=0.0):
    p1, p2 = G[k]["p1"], G[k]["p2"]
    pts = p1 + np.linspace(0, 1, n_pts)[:, None] * (p2 - p1)
    keep = (np.linalg.norm(pts - p1, axis=1) >= clearance) & (np.linalg.norm(pts - p2, axis=1) >= clearance)
    return pts[keep]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
rows = []
for k in range(n - 1):
    nxt = k + 1
    d_cross = np.array([gate_distance(p, nxt) for p in gate_points(k, clearance=PYLON_FIELD_R_M)])
    rows.append({"stage": k, "target G_k": G[k]["name"], "avoid G_k+1": G[nxt]["name"],
                 "length of G_k (m)": G[k]["length"],
                 "min(d_G_k+1) (m)": d_cross.min(),
                 "max(d_G_k+1) (m)": d_cross.max()})
stage_table = pd.DataFrame(rows)
display(stage_table.round(2))

for _, row in stage_table.iterrows():
    print(f"stage {row['stage']} ({row['target G_k']} -> avoid {row['avoid G_k+1']}): "
          f"min(d_G_k+1) = {row['min(d_G_k+1) (m)']:.2f} m, max(d_G_k+1) = {row['max(d_G_k+1) (m)']:.2f} m")

,stage,target G_k,avoid G_k+1,length of G_k (m),min(d_G_k+1) (m),max(d_G_k+1) (m)
0,0,G1,G2,25.00,2.50,22.50
1,1,G2,G3,25.50,2.05,18.76
2,2,G3,G4,49.24,2.51,46.73
3,3,G4,G5,29.15,1.86,19.72
4,4,G5,G6,25.50,2.51,22.98


stage 0 (G1 -> avoid G2): min(d_G_k+1) = 2.50 m, max(d_G_k+1) = 22.50 m
stage 1 (G2 -> avoid G3): min(d_G_k+1) = 2.05 m, max(d_G_k+1) = 18.76 m
stage 2 (G3 -> avoid G4): min(d_G_k+1) = 2.51 m, max(d_G_k+1) = 46.73 m
stage 3 (G4 -> avoid G5): min(d_G_k+1) = 1.86 m, max(d_G_k+1) = 19.72 m
stage 4 (G5 -> avoid G6): min(d_G_k+1) = 2.51 m, max(d_G_k+1) = 22.98 m


In [40]:
first_stage = 0      

d0 = gate_distance(START_MAP, 1)
print(f"Distance from START to G2 (stage 0): {d0:.1f} m")

Distance from START to G2 (stage 0): 13.5 m


In [41]:
center_rows = []
for k in range(first_stage, n - 1):
    center = 0.5 * (G[k]["p1"] + G[k]["p2"])
    center_rows.append({"gate crossed (G_k)": G[k]["name"],
                        "gate to avoid (G_k+1)": G[k + 1]["name"],
                        "distance from the center of G_k to G_k+1 (m)": gate_distance(center, k + 1)})
center_table = pd.DataFrame(center_rows)
display(center_table.round(2))

print(f"Smallest center distance: {center_table['distance from the center of G_k to G_k+1 (m)'].min():.2f} m")

,gate crossed (G_k),gate to avoid (G_k+1),distance from the center of G_k to G_k+1 (m)
0,G1,G2,12.50
1,G2,G3,10.41
2,G3,G4,24.62
3,G4,G5,10.79
4,G5,G6,12.75


Smallest center distance: 10.41 m


Note:
- max(min(d_G_k+1)) = 2.51 m
- min(max(d_G_k+1)) = 18.76 m
- r_2 < 13.5 m (stage 0)
- r_2 < 10.41 m (distance between gates to avoid)

-> r_2 must be between **(3, 10)**

In [42]:
R2_CANDIDATES = [float(r) for r in range(3, 11)]

check = []
for r2 in R2_CANDIDATES:
    row = {"r2 (m)": r2, "START outside r2 (stage 0 usable)": bool(r2 < d0)}
    for k in range(first_stage, n - 1):
        d = np.array([gate_distance(p, k + 1) for p in gate_points(k, clearance=PYLON_FIELD_R_M)])
        row[f"% of {G[k]['name']} penalized (avoid {G[k + 1]['name']})"] = 100 * np.mean(d < r2)
    check.append(row)
r2_table = pd.DataFrame(check)
display(r2_table.round(0))

,r2 (m),START outside r2 (stage 0 usable),% of G1 penalized (avoid G2),% of G2 penalized (avoid G3),% of G3 penalized (avoid G4),% of G4 penalized (avoid G5),% of G5 penalized (avoid G6)
0,3.0,True,2.0,6.0,1.0,6.0,2.0
1,4.0,True,7.0,12.0,3.0,12.0,7.0
2,5.0,True,12.0,18.0,6.0,18.0,12.0
3,6.0,True,17.0,24.0,8.0,23.0,17.0
4,7.0,True,22.0,30.0,10.0,29.0,22.0
5,8.0,True,27.0,36.0,12.0,34.0,27.0
6,9.0,True,32.0,42.0,15.0,40.0,32.0
7,10.0,True,37.0,48.0,17.0,46.0,37.0
